In [ ]:
from datasets import load_dataset
from itertools import islice

dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.simple",
    split="train",
    streaming=True
)

articles = list(islice(dataset, 500))

print("Loaded:", len(articles))
print("Columns:", articles[0].keys())

for article in articles:
    print("\n" + "=" * 60)
    print("TITLE:", article["title"])
    print("TEXT:")
    print(article["text"][:1500])


In [ ]:
# Cell 2: Inspect full articles

titles = [
    "April",
    "Art",
    "Alan Turing",
    "Adobe Illustrator"
]

for title in titles:
    article = next(
        a for a in articles
        if a["title"] == title
    )

    print("\n" + "=" * 70)
    print("ARTICLE:", title)
    print("=" * 70)

    lines = article["text"].splitlines()

    for i, line in enumerate(lines, 1):
        print(f"{i:4}: {line}")

    print("\nTOTAL LINES:", len(lines))


In [ ]:
from pathlib import Path
import sys

for folder in [
    Path.cwd(),
    Path.cwd() / "backend/notebooks/modern"
]:
    if (folder / "modern_english_cleaning.py").exists():
        sys.path.insert(0, str(folder))
        break

from modern_english_cleaning import (
    clean_article,
    validate_article
)

for article in articles[:4]:
    cleaned, report = clean_article(
        article["text"],
        title=article["title"]
    )

    flags = validate_article(cleaned, report)

    print("=" * 60)
    print("ARTICLE:", article["title"])
    print("REMOVED:", report["removed_pct"], "%")
    print("FLAGS:", flags)
    print("\nCLEANED TEXT:")
    print(cleaned[:1000])

In [ ]:
# Inspect the endings of cleaned articles

for article in articles[:10]:
    cleaned, report = clean_article(
        article["text"],
        title=article["title"]
    )

    flags = validate_article(cleaned, report)

    print("=" * 60)
    print("ARTICLE:", article["title"])

    print("\nORIGINAL END:")
    print(article["text"][-500:])

    print("\nCLEANED END:")
    print(cleaned[-500:])

    print("\nREMOVED:", report["removed_pct"], "%")
    print("FLAGS:", flags)


In [ ]:

from collections import Counter

results = []

for article in articles:
    cleaned, report = clean_article(
        article["text"],
        title=article["title"]
    )

    flags = validate_article(cleaned, report)

    results.append({
        "title": article["title"],
        "original": article["text"],
        "cleaned": cleaned,
        "removed_pct": report["removed_pct"],
        "flags": flags
    })

print("Total articles:", len(results))
print("Flagged articles:", sum(bool(r["flags"]) for r in results))
print("Empty articles:", sum(not r["cleaned"] for r in results))

flag_counts = Counter(
    flag for r in results for flag in r["flags"]
)

print("\nFLAG COUNTS:")
for flag, count in flag_counts.most_common():
    print(f"{flag}: {count}")

print("\nTOP 10 MOST MODIFIED:")

most_modified = sorted(
    results,
    key=lambda r: r["removed_pct"],
    reverse=True
)

for article in most_modified[:10]:
    print(
        f"{article['title'][:35]:35} "
        f"{article['removed_pct']:.2f}% "
        f"{article['flags']}"
    )


In [ ]:

# Cell 4: Investigate heavily cleaned articles

titles_to_check = [
    "Historian",
    "Healing",
    "May",
    "List of mathematics topics",
    "Southern Ocean",
    "Saint Lawrence River",
    "Indian Ocean"
]

for title in titles_to_check:
    article = next(
        (a for a in articles if a["title"] == title),
        None
    )

    if article is None:
        continue

    original = article["text"]

    cleaned, report = clean_article(
        original,
        title=title
    )

    print("\n" + "=" * 70)
    print("ARTICLE:", title)
    print("REMOVED:", report["removed_pct"], "%")
    print("STOPPED AT:", report["stopped_at_back_matter"])

    print("\n=== END OF CLEANED TEXT ===")
    print(cleaned[-700:])

    # Find where the cleaning stopped
    lines = original.splitlines()
    section = report["stopped_at_back_matter"]

    if section:
        index = next(
            (
                i for i, line in enumerate(lines)
                if line.strip().rstrip(":").casefold()
                == section
            ),
            None
        )

        if index is not None:
            print("\n=== ORIGINAL AROUND STOP POINT ===")

            start = max(0, index - 5)
            end = min(len(lines), index + 15)

            for j in range(start, end):
                print(f"{j+1}: {lines[j][:200]}")

            print(
                "\nLINES AFTER STOP:",
                len(lines) - index - 1
            )

    print("\n=== ORIGINAL END ===")
    print(original[-700:])


In [ ]:
import importlib
import modern_english_cleaning

importlib.reload(modern_english_cleaning)

clean_article = modern_english_cleaning.clean_article

for title in ["May", "Historian", "Healing"]:
    article = next(
        a for a in articles if a["title"] == title
    )

    cleaned, report = clean_article(
        article["text"],
        title=title
    )

    print("\nARTICLE:", title)
    print("REMOVED:", report["removed_pct"], "%")

    if title == "May":
        print("Historical events preserved:",
              "Great Exhibition" in cleaned)

    if title == "Historian":
        print("References removed:",
              "Dictionary of British Classicists" not in cleaned)

In [ ]:
from collections import Counter

results = []

for article in articles:
    cleaned, report = clean_article(
        article["text"],
        title=article["title"]
    )

    flags = validate_article(cleaned, report)

    results.append({
        "title": article["title"],
        "original": article["text"],
        "cleaned": cleaned,
        "removed_pct": report["removed_pct"],
        "flags": flags
    })

print("Total articles:", len(results))
print("Flagged articles:", sum(bool(r["flags"]) for r in results))
print("Empty articles:", sum(not r["cleaned"] for r in results))

print("\nFLAG COUNTS:")
for flag, count in Counter(
    flag for r in results for flag in r["flags"]
).most_common():
    print(flag, count)

print("\nTOP 10 MOST MODIFIED:")
for r in sorted(
    results,
    key=lambda x: x["removed_pct"],
    reverse=True
)[:10]:
    print(
        f"{r['title'][:35]:35} "
        f"{r['removed_pct']:.2f}% "
        f"{r['flags']}"
    )

In [ ]:

# Inspect remaining cleaning problems

import re

# Patterns for leftover Wikipedia markup
MARKUP_PATTERN = re.compile(
    r"\[\[|\]\]|\{\{|\}\}|<ref\b",
    re.I
)

for result in results:

    relevant_flags = {
        "possible_trailing_metadata",
        "leftover_wiki_markup"
    }

    if not relevant_flags.intersection(result["flags"]):
        continue

    print("\n" + "=" * 65)
    print("ARTICLE:", result["title"])
    print("FLAGS:", result["flags"])
    print("REMOVED:", result["removed_pct"], "%")

    print("\n=== ORIGINAL LAST 8 LINES ===")
    print("\n".join(result["original"].splitlines()[-8:]))

    print("\n=== CLEANED LAST 8 LINES ===")
    print("\n".join(result["cleaned"].splitlines()[-8:]))

    # Show remaining markup and its context
    matches = list(MARKUP_PATTERN.finditer(result["cleaned"]))

    if matches:
        print("\n=== LEFTOVER MARKUP ===")

        for match in matches[:5]:
            start = max(0, match.start() - 60)
            end = min(len(result["cleaned"]), match.end() + 60)

            print(repr(result["cleaned"][start:end]))


In [ ]:
import importlib
import modern_english_cleaning

importlib.reload(modern_english_cleaning)

clean_article = modern_english_cleaning.clean_article

for title in ["April", "May", "December"]:
    article = next(a for a in articles if a["title"] == title)
    cleaned, report = clean_article(article["text"], title=title)

    print(f"\nARTICLE: {title}")
    print("Last line:", repr(cleaned.splitlines()[-1]))
    print("Removed:", report["removed_pct"], "%")

In [ ]:
import re

titles = [
    "Embassy",
    "Earth science",
    "Experiment",
    "South America"
]

for title in titles:
    article = next(a for a in articles if a["title"] == title)
    original = article["text"]

    print("\n" + "=" * 60)
    print("ARTICLE:", title)

    matches = list(re.finditer(
        r"\[\[|<ref\b", original, re.I
    ))

    for match in matches[:5]:
        start = max(0, match.start() - 100)
        end = min(len(original), match.start() + 500)

        print("\nMARKUP CONTEXT:")
        print(repr(original[start:end]))

In [ ]:
import importlib
import modern_english_cleaning

importlib.reload(modern_english_cleaning)

clean_article = modern_english_cleaning.clean_article
validate_article = modern_english_cleaning.validate_article

for title in [
    "Earth science",
    "Experiment",
    "South America",
    "Embassy"
]:
    article = next(
        a for a in articles if a["title"] == title
    )

    cleaned, report = clean_article(
        article["text"],
        title=title
    )

    flags = validate_article(cleaned, report)

    print("\nARTICLE:", title)
    print("Remaining <ref>:", "<ref" in cleaned.lower())
    print("FLAGS:", flags)

In [ ]:

import importlib
import modern_english_cleaning
from collections import Counter

# Reload the latest cleaning functions
importlib.reload(modern_english_cleaning)

clean_article = modern_english_cleaning.clean_article
validate_article = modern_english_cleaning.validate_article

results = []

for article in articles:
    cleaned, report = clean_article(
        article["text"],
        title=article["title"]
    )

    flags = validate_article(cleaned, report)

    results.append({
        "title": article["title"],
        "original": article["text"],
        "cleaned": cleaned,
        "removed_pct": report["removed_pct"],
        "flags": flags
    })

print("TOTAL ARTICLES:", len(results))
print("FLAGGED:", sum(bool(r["flags"]) for r in results))
print("EMPTY:", sum(not r["cleaned"] for r in results))

print("\nFLAG COUNTS:")
counts = Counter(
    flag for r in results for flag in r["flags"]
)

for flag, count in counts.most_common():
    print(f"{flag}: {count}")

print("\nREMAINING MARKUP ISSUES:")
for r in results:
    if "leftover_wiki_markup" in r["flags"]:
        print(r["title"])

# Verify meaningful content was preserved
south_america = next(
    r for r in results
    if r["title"] == "South America"
)

print("\nSOUTH AMERICA CHECK:")
print(
    "Panama Canal sentence preserved:",
    "These two continents are separated by the Panama Canal."
    in south_america["cleaned"]
)



In [ ]:

# Final cleaning review

flagged = [r for r in results if r["flags"]]

print("FLAGGED ARTICLES:", len(flagged))

for r in flagged:
    print("\n" + "=" * 60)
    print("TITLE:", r["title"])
    print("FLAGS:", r["flags"])
    print("REMOVED:", r["removed_pct"], "%")

    print("\nCLEANED TEXT (last 400 chars):")
    print(r["cleaned"][-400:])


# Check important known edge cases
print("\n" + "=" * 60)
print("KNOWN EDGE CASES")

for title in ["Andouille", "Air", "May", "Embassy"]:
    r = next(
        (item for item in results if item["title"] == title),
        None
    )

    if r is None:
        continue

    print("\nARTICLE:", title)
    print("FLAGS:", r["flags"])
    print("LAST 3 LINES:")
    print("\n".join(r["cleaned"].splitlines()[-3:]))


# Regression checks
may = next(r for r in results if r["title"] == "May")
south = next(r for r in results if r["title"] == "South America")

print("\nREGRESSION CHECKS")
print("May historical events preserved:",
      "Great Exhibition" in may["cleaned"])

print("South America Panama sentence preserved:",
      "These two continents are separated by the Panama Canal."
      in south["cleaned"])


In [ ]:
import importlib
import modern_english_cleaning

importlib.reload(modern_english_cleaning)

for title in ["Andouille", "Embassy", "Cost of living", "Italians"]:
    article = next(a for a in articles if a["title"] == title)

    cleaned, report = modern_english_cleaning.clean_article(
        article["text"],
        title=title
    )

    flags = modern_english_cleaning.validate_article(cleaned, report)

    print(title, ":", flags)

In [ ]:
article = next(
    a for a in articles
    if a["title"] == "Cost of living"
)

cleaned, report = clean_article(
    article["text"],
    title=article["title"]
)

print("ORIGINAL:")
print(repr(article["text"]))

print("\nCLEANED LINES:")
for i, line in enumerate(cleaned.splitlines(), 1):
    print(i, repr(line))

In [ ]:
import importlib
import modern_english_cleaning

importlib.reload(modern_english_cleaning)

for title in ["Cost of living", "Andouille", "Embassy", "Italians"]:
    article = next(a for a in articles if a["title"] == title)

    cleaned, report = modern_english_cleaning.clean_article(
        article["text"],
        title=title
    )

    flags = modern_english_cleaning.validate_article(
        cleaned, report
    )

    print(title, ":", flags)

In [ ]:

import importlib
from collections import Counter
import modern_english_cleaning

importlib.reload(modern_english_cleaning)

clean_article = modern_english_cleaning.clean_article
validate_article = modern_english_cleaning.validate_article

results = []

for article in articles:
    cleaned, report = clean_article(
        article["text"],
        title=article["title"]
    )

    flags = validate_article(cleaned, report)

    results.append({
        "title": article["title"],
        "original": article["text"],
        "cleaned": cleaned,
        "report": report,
        "flags": flags
    })

print("TOTAL:", len(results))
print("FLAGGED:", sum(bool(r["flags"]) for r in results))
print("EMPTY:", sum(not r["cleaned"] for r in results))

print("\nFLAG COUNTS:")
for flag, count in Counter(
    flag for r in results for flag in r["flags"]
).most_common():
    print(f"{flag}: {count}")

print("\nPOSSIBLE TRAILING CATEGORIES:")
for r in results:
    if "possible_trailing_category" in r["flags"]:
        print(
            f"{r['title']:35} -> "
            f"{r['cleaned'].splitlines()[-1]}"
        )

print("\nREGRESSION CHECKS:")
checks = {
    "May events preserved": "Great Exhibition" in next(
        r["cleaned"] for r in results if r["title"] == "May"
    ),
    "South America sentence preserved": "These two continents are separated by the Panama Canal." in next(
        r["cleaned"] for r in results if r["title"] == "South America"
    )
}

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")


In [ ]:

# Investigate empty parentheses in cleaned articles

import re

empty_parens = [
    r for r in results
    if "empty_parentheses" in r["flags"]
]

print("ARTICLES WITH EMPTY PARENTHESES:", len(empty_parens))

original_issue = 0
introduced_by_cleaning = 0

for article in empty_parens:
    original = article["original"]
    cleaned = article["cleaned"]

    original_has = bool(re.search(r"\(\s*\)", original))

    if original_has:
        original_issue += 1
        source = "ALREADY IN ORIGINAL"
    else:
        introduced_by_cleaning += 1
        source = "POSSIBLY INTRODUCED BY CLEANING"

    print("\n" + "=" * 60)
    print("ARTICLE:", article["title"])
    print("SOURCE:", source)

    match = re.search(r"\(\s*\)", cleaned)

    if match:
        start = max(0, match.start() - 100)
        end = min(len(cleaned), match.end() + 100)

        print("CONTEXT:")
        print(repr(cleaned[start:end]))

print("\n" + "=" * 60)
print("Already in original:", original_issue)
print("Possibly introduced by cleaning:", introduced_by_cleaning)


In [ ]:
from pathlib import Path
import json

OUTPUT_DIR = Path("modern_cleaning_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# High-priority review flags
REVIEW_FLAGS = {
    "empty_parentheses",
    "leftover_wiki_markup"
}

review_count = 0

with (
    open(OUTPUT_DIR / "cleaned_articles.jsonl",
         "w", encoding="utf-8") as all_file,
    open(OUTPUT_DIR / "needs_review.jsonl",
         "w", encoding="utf-8") as review_file
):
    for article in results:
        record = {
            "title": article["title"],
            "text": article["cleaned"],
            "flags": article["flags"]
        }

        line = json.dumps(record, ensure_ascii=False) + "\n"
        all_file.write(line)

        if REVIEW_FLAGS.intersection(article["flags"]):
            review_file.write(line)
            review_count += 1

print("Total exported:", len(results))
print("Priority review articles:", review_count)
print("Saved to:", OUTPUT_DIR.resolve())